In [3]:
from db_connection import setup_sakila, save_result_csv

engine = setup_sakila(displaylimit=None)

displaylimit: Value None will be treated as 0 (no limit)

## 1. テーブルの列情報を確認する

In [5]:
%%sql

show columns from customer;

Field,Type,Null,Key,Default,Extra
customer_id,smallint unsigned,NO,PRI,None,auto_increment
store_id,tinyint unsigned,NO,MUL,None,
first_name,varchar(45),NO,,None,
last_name,varchar(45),NO,MUL,None,
email,varchar(50),YES,,None,
address_id,smallint unsigned,NO,MUL,None,
active,tinyint(1),NO,,1,
create_date,datetime,NO,,None,
last_update,timestamp,YES,,CURRENT_TIMESTAMP,DEFAULT_GENERATED on update CURRENT_TIMESTAMP


## 2. SELECT句

- 1つの列を照会する

In [7]:
%%sql

SELECT first_name
FROM customer
LIMIT 5;

first_name
MARY
PATRICIA
LINDA
BARBARA
ELIZABETH


- 2つ以上の列を照会する (列を指定する順序は関係ない)

In [8]:
%%sql

SELECT first_name, last_name
FROM customer
LIMIT 5;

first_name,last_name
MARY,SMITH
PATRICIA,JOHNSON
LINDA,WILLIAMS
BARBARA,JONES
ELIZABETH,BROWN


In [ ]:
%%sql

SELECT last_name, first_name
FROM customer
LIMIT 5;

# 照会する列の順序は、元のテーブルの列順とは関係ない

last_name,first_name
SMITH,MARY
JOHNSON,PATRICIA
WILLIAMS,LINDA
JONES,BARBARA
BROWN,ELIZABETH


- すべての列を照会する

In [10]:
%%sql

SELECT *
FROM customer
LIMIT 5;

customer_id,store_id,first_name,last_name,email,address_id,active,create_date,last_update
1,1,MARY,SMITH,MARY.SMITH@sakilacustomer.org,5,1,2006-02-14 22:04:36,2006-02-15 04:57:20
2,1,PATRICIA,JOHNSON,PATRICIA.JOHNSON@sakilacustomer.org,6,1,2006-02-14 22:04:36,2006-02-15 04:57:20
3,1,LINDA,WILLIAMS,LINDA.WILLIAMS@sakilacustomer.org,7,1,2006-02-14 22:04:36,2006-02-15 04:57:20
4,2,BARBARA,JONES,BARBARA.JONES@sakilacustomer.org,8,1,2006-02-14 22:04:36,2006-02-15 04:57:20
5,1,ELIZABETH,BROWN,ELIZABETH.BROWN@sakilacustomer.org,9,1,2006-02-14 22:04:36,2006-02-15 04:57:20


## 3. DISTINCT

- `DISTINCT`は、`SELECT`した結果から重複した行を除外し、一意の値だけを取得するために使用する。

- `DISTINCT`は、論理的には`SELECT`句の処理後に適用される。<br> (FROM -> WHERE -> SELECT -> DISTINCT -> ORDER BY -> LIMIT)


#### # 1つの列に`DISTINCT`を使用する場合

```sql
SELECT DISTINCT カラム名
FROM テーブル名;
```

#### # 2つ以上の列に`DISTINCT`を使用する場合

```sql
SELECT DISTINCT カラム名1, カラム名2
FROM テーブル名;
```

In [11]:
%%sql

SELECT DISTINCT store_id
FROM customer;

store_id
1
2


In [13]:
%%sql

SELECT DISTINCT store_id, active
FROM customer;

store_id,active
1,1
2,1
2,0
1,0


In [22]:
%%sql

SELECT DISTINCT first_name
FROM customer
ORDER BY first_name
LIMIT 5;


first_name
AARON
ADAM
ADRIAN
AGNES
ALAN


#### ※ `DISTINCT`の注意点

`DISTINCT`は、各カラムを個別に重複除外するのではなく、`SELECT`したカラムの組み合わせ全体を基準に重複を除外する。

```sql
- store_idとactiveの組み合わせが同じ行は、1行だけ取得する。
SELECT DISTINCT store_id, active
FROM customer;
```

例えば、元のデータが次のような場合：

| store_id | active |
|---|---|
| 1 | 1 |
| 1 | 1 |
| 1 | 0 |
| 2 | 1 |
| 2 | 1 |

`SELECT DISTINCT store_id, active`の実行結果は次のようになる。

| store_id | active |
|---|---|
| 1 | 1 |
| 1 | 0 |
| 2 | 1 |